In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/divyanshikumari28/hindi-dataset/combined_hindi_3class_dataset_fixed.csv


In [2]:
!pip -q install -U transformers accelerate scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 41.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires panda

In [3]:
import torch
import transformers
import accelerate
from accelerate import Accelerator

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU available:", torch.cuda.is_available())

acc = Accelerator()

print("Accelerator state:", vars(acc.state).keys())
print("Distributed type:", acc.state.distributed_type)
print("Device:", acc.device)

PyTorch: 2.11.0+cu128
Transformers: 5.19.0
Accelerate: 1.15.0
GPU available: True
Accelerator state: dict_keys(['_cpu', 'backend', 'device', 'debug', 'distributed_type', 'num_processes', 'process_index', 'local_process_index', 'fork_launched', 'deepspeed_plugins', 'torch_tp_plugin', 'parallelism_config', 'device_mesh', 'dynamo_plugin', '_mixed_precision'])
Distributed type: DistributedType.NO
Device: cuda


In [4]:
import os
import glob
import random
import inspect
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from torch.nn import CrossEntropyLoss

In [5]:
# --------------------------------------------------
# 1. Reproducibility and GPU check
# --------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Enable GPU in Kaggle notebook settings."
    )

GPU: Tesla T4


In [6]:
# --------------------------------------------------
# 2. Find the combined CSV
# --------------------------------------------------
matches = glob.glob(
    "/kaggle/input/**/combined_hindi_3class_dataset_fixed.csv",
    recursive=True
)

if not matches:
    raise FileNotFoundError(
        "Add combined_hindi_3class_dataset_fixed.csv "
        "using Kaggle's Add Input option."
    )

DATA_PATH = matches[0]
df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# Automatically identify columns
def find_col(candidates, columns):
    lookup = {str(c).lower(): c for c in columns}
    for name in candidates:
        if name in lookup:
            return lookup[name]
    return None

CLAIM_COL = find_col(
    ["claim", "claim_text", "text", "statement",
     "claim_text_hi", "claim_hi", "input_text"],
    df.columns
)

LABEL_COL = find_col(
    ["label_id", "label", "label_name", "verdict",
     "class", "target", "gold_label"],
    df.columns
)
# If auto-detection is wrong, replace these with your actual columns.
print("Detected claim column:", CLAIM_COL)
print("Detected label column:", LABEL_COL)

assert CLAIM_COL is not None, "Set CLAIM_COL manually."
assert LABEL_COL is not None, "Set LABEL_COL manually."

Dataset path: /kaggle/input/datasets/divyanshikumari28/hindi-dataset/combined_hindi_3class_dataset_fixed.csv
Original shape: (5980, 12)
Columns: ['claim', 'label_raw', 'label_id', 'language', 'source_dataset', 'title', 'evidence', 'link', 'dataset_origin', 'label_status', 'claim_key', 'label_name']
Detected claim column: claim
Detected label column: label_id


In [7]:
# --------------------------------------------------
# 3. Clean data and preserve label meanings
# --------------------------------------------------
LABEL2ID = {
    "false": 0,
    "mostly false": 1,
    "partly true/misleading": 2
}
ID2LABEL = {
    0: "False",
    1: "Mostly False",
    2: "Partly True/Misleading"
}

data = df[[CLAIM_COL, LABEL_COL]].copy()
data.columns = ["claim", "raw_label"]
data = data.dropna(subset=["claim", "raw_label"])
data["claim"] = data["claim"].astype(str).str.strip()
data = data[data["claim"].str.len() > 0].copy()

numeric = pd.to_numeric(data["raw_label"], errors="coerce")

if numeric.notna().all():
    data["label"] = numeric.astype(int)
else:
    labels = data["raw_label"].astype(str).str.strip().str.lower()
    mapped = labels.map(LABEL2ID)
    if mapped.isna().any():
        raise ValueError(
            "Unknown labels: " +
            str(labels[mapped.isna()].unique().tolist())
        )
    data["label"] = mapped.astype(int)

assert set(data["label"].unique()).issubset({0, 1, 2})

# Remove conflicting duplicates; do not silently choose a label.
conflicts = data.groupby("claim")["label"].nunique()
conflicts = conflicts[conflicts > 1]

if len(conflicts):
    raise ValueError(
        f"{len(conflicts)} claims have conflicting labels. "
        "Resolve them before training."
    )

data = data.drop_duplicates("claim").reset_index(drop=True)
data["label"] = data["label"].astype(int)

print("Cleaned rows:", len(data))
print(data["label"].value_counts().sort_index())

Cleaned rows: 1936
label
0    1256
1     389
2     291
Name: count, dtype: int64


In [8]:
# --------------------------------------------------
# 4. Stratified 80/10/10 split
# --------------------------------------------------
train_df, temp_df = train_test_split(
    data, test_size=0.20, random_state=SEED,
    stratify=data["label"]
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1548
Validation: 194
Test: 194


In [9]:
# --------------------------------------------------
# 5. Load GOOGLE MURILL-CASED and tokenize claims
# --------------------------------------------------
MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 256
OUTPUT_DIR = "/kaggle/working/muril_combined_1936"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id={v: k for k, v in ID2LABEL.items()}
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config
)

def tokenize_batch(batch):
    return tokenizer(
        batch["claim"],
        truncation=True,
        max_length=MAX_LENGTH
    )

def make_dataset(frame):
    frame = frame[["claim", "label"]].rename(
        columns={"label": "labels"}
    )
    ds = Dataset.from_pandas(frame, preserve_index=False)
    return ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=["claim"]
    )

train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

...

Map:   0%|          | 0/1548 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

In [10]:
# --------------------------------------------------
# 6. Metrics
# --------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(
            labels, preds, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            labels, preds, average="weighted", zero_division=0
        )
    }
    # Class weights calculated from training data only
counts = (
    train_df["label"].value_counts()
    .reindex([0, 1, 2], fill_value=0)
    .sort_index()
)
weights = len(train_df) / (
    3 * torch.tensor(counts.values, dtype=torch.float32)
)
print("Training class weights:", weights.tolist())

Training class weights: [0.5139442682266235, 1.6591639518737793, 2.214592218399048]


In [11]:
import inspect

args_dict = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,      # use 8 if you hit OOM
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,        # was 1e-5
    warmup_steps=50,           # was 100
    num_train_epochs=8,        # nothing improves after about 6 to 8 anyway
    weight_decay=0.01,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=True,
    dataloader_num_workers=0,
)

# Handle both old and new transformers argument names
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_dict)
print("training_args created")

training_args created


In [12]:
from transformers import EarlyStoppingCallback

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [13]:
from torch.nn import CrossEntropyLoss
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self, model, inputs, return_outputs=False,
        num_items_in_batch=None
    ):
        inputs = inputs.copy()
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        base_model = model.module if hasattr(model, "module") else model
        num_labels = base_model.config.num_labels

        loss_fn = CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )
        loss = loss_fn(
            outputs.logits.reshape(-1, num_labels),
            labels.reshape(-1)
        )

        return (loss, outputs) if return_outputs else loss


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=weights
)

print("Trainer created.")
print("Device:", trainer.accelerator.device)

Trainer created.
Device: cuda


In [14]:
# --------------------------------------------------
# 9. Train
# --------------------------------------------------
trainer.train()

# Save selected best model and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved:", OUTPUT_DIR)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro F1:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.090185,1.075409,0.649485,0.455299,0.613852
2,0.994895,0.977541,0.649485,0.453724,0.612903
3,0.923013,0.926606,0.659794,0.460472,0.621779
4,0.854755,0.919900,0.649485,0.451407,0.611920
5,0.816506,0.920474,0.664948,0.479925,0.632707
6,0.781596,0.926804,0.680412,0.552916,0.672876
7,0.748043,0.932300,0.675258,0.551249,0.667588
8,0.719106,0.935346,0.675258,0.561959,0.672026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved: /kaggle/working/muril_combined_1936
Best checkpoint: /kaggle/working/muril_combined_1936/checkpoint-392
Best validation macro F1: 0.5619592615300769


In [15]:
val_out = trainer.predict(val_ds)
val_logits, y_val = val_out.predictions, val_out.label_ids

best_bias, best_f1 = (0.0, 0.0), -1
for b1 in np.arange(-1, 1.01, 0.25):
    for b2 in np.arange(0, 3.01, 0.25):
        adj = val_logits + np.array([0, b1, b2])
        f1 = f1_score(y_val, adj.argmax(1), average="macro", zero_division=0)
        if f1 > best_f1:
            best_bias, best_f1 = (b1, b2), f1
bias = np.array([0, *best_bias])
print("Best bias:", best_bias, "| val macro F1:", round(best_f1, 4))

test_result = trainer.predict(test_ds)
test_logits = test_result.predictions + bias     # use this instead of the raw logits
preds = np.argmax(test_logits, axis=-1)
true = test_result.label_ids

Best bias: (np.float64(-0.25), np.float64(0.0)) | val macro F1: 0.5786


In [16]:
assert (test_df["label"].values == true).all(), "test_df and test_ds are misaligned"

In [17]:
!pip -q install rank_bm25 sentence-transformers

In [18]:
import os, re, glob, shutil, joblib
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
 
SEED = 42
ART = "/kaggle/working/artifacts"
MURIL_DIR = "/kaggle/working/muril_combined_1936"      # your MuRIL output dir
SBERT_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
ID2LABEL = {0: "False", 1: "Mostly False", 2: "Partly True/Misleading"}
os.makedirs(ART, exist_ok=True)

In [19]:
# --------------------------------------------------
# 1. Data: same labeled subset and split as the MuRIL run
#    (if you used the origin+label stratify patch, stratify the same way here)
# --------------------------------------------------
DATA_PATH = glob.glob("/kaggle/input/**/combined_hindi_3class_dataset_fixed.csv", recursive=True)[0]
df = pd.read_csv(DATA_PATH)
 
lab = df[df["label_id"].notna()].copy()
lab["label"] = lab["label_id"].astype(int)
lab["claim"] = lab["claim"].astype(str).str.strip()
lab = lab.drop_duplicates("claim").reset_index(drop=True)
 
train_df, temp_df = train_test_split(lab, test_size=0.20, random_state=SEED, stratify=lab["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])
print("train/val/test:", len(train_df), len(val_df), len(test_df))
 
ytr, yva, yte = train_df["label"].values, val_df["label"].values, test_df["label"].values
prior = np.bincount(ytr, minlength=3) / len(ytr)
 
def report(name, y_true, y_pred):
    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4),
          "| Macro F1:", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print(classification_report(y_true, y_pred, labels=[0, 1, 2],
          target_names=[ID2LABEL[i] for i in range(3)], digits=4, zero_division=0))
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

train/val/test: 1548 194 194


In [20]:
# --------------------------------------------------
# 2. BM25
# --------------------------------------------------
# Python's \w splits Devanagari words at vowel signs, so use explicit ranges
# (U+0900-U+097F, excluding the danda marks U+0964/U+0965).
TOKEN_RE = re.compile(r"[\u0900-\u0963\u0966-\u097F\w]+")
def tok(text):
    return TOKEN_RE.findall(str(text).lower())
 
bm25_train = BM25Okapi([tok(t) for t in train_df["claim"]])
 
def vote(scores, labels, k):
    idx = np.argsort(-scores)[:k]
    v = np.zeros(3)
    for i in idx:
        v[labels[i]] += max(scores[i], 0.0)
    return v / prior          # divide by class prior to counter imbalance
 
def bm25_knn_predict(texts, k):
    return np.array([vote(bm25_train.get_scores(tok(t)), ytr, k).argmax() for t in texts])
 
best_k, best_f1 = 5, -1
for k in [3, 5, 10, 20, 50]:
    f1 = f1_score(yva, bm25_knn_predict(val_df["claim"], k), average="macro", zero_division=0)
    print(f"BM25 kNN k={k}: val macro F1 {f1:.4f}")
    if f1 > best_f1:
        best_k, best_f1 = k, f1
report(f"BM25 kNN (k={best_k}) - TEST", yte, bm25_knn_predict(test_df["claim"], best_k))

BM25 kNN k=3: val macro F1 0.4279
BM25 kNN k=5: val macro F1 0.4003
BM25 kNN k=10: val macro F1 0.3903
BM25 kNN k=20: val macro F1 0.4010
BM25 kNN k=50: val macro F1 0.3543

=== BM25 kNN (k=3) - TEST ===
Accuracy: 0.6443 | Macro F1: 0.5175
                        precision    recall  f1-score   support

                 False     0.7795    0.7857    0.7826       126
          Mostly False     0.5000    0.3077    0.3810        39
Partly True/Misleading     0.3256    0.4828    0.3889        29

              accuracy                         0.6443       194
             macro avg     0.5350    0.5254    0.5175       194
          weighted avg     0.6555    0.6443    0.6430       194

[[99  8 19]
 [17 12 10]
 [11  4 14]]


In [21]:
# --------------------------------------------------
# 3. SBERT: embeddings + kNN + LogisticRegression
# --------------------------------------------------
sbert = SentenceTransformer(SBERT_NAME, device="cuda")
sbert.max_seq_length = 256
 
def enc(texts):
    return sbert.encode(list(texts), batch_size=64, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True)
 
Xtr, Xva, Xte = enc(train_df["claim"]), enc(val_df["claim"]), enc(test_df["claim"])
 
def sbert_knn_predict(Xq, k):
    sims = Xq @ Xtr.T
    return np.array([vote(s, ytr, k).argmax() for s in sims])
 
best_ks, best_f1s = 5, -1
for k in [3, 5, 10, 20, 50]:
    f1 = f1_score(yva, sbert_knn_predict(Xva, k), average="macro", zero_division=0)
    print(f"SBERT kNN k={k}: val macro F1 {f1:.4f}")
    if f1 > best_f1s:
        best_ks, best_f1s = k, f1
report(f"SBERT kNN (k={best_ks}) - TEST", yte, sbert_knn_predict(Xte, best_ks))
 
best_C, best_f1c = 1.0, -1
for C in [0.1, 1, 10, 100]:
    lr = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(Xtr, ytr)
    f1 = f1_score(yva, lr.predict(Xva), average="macro", zero_division=0)
    print(f"SBERT + LR C={C}: val macro F1 {f1:.4f}")
    if f1 > best_f1c:
        best_C, best_f1c = C, f1
lr = LogisticRegression(C=best_C, max_iter=3000, class_weight="balanced").fit(Xtr, ytr)
report(f"SBERT + LogisticRegression (C={best_C}) - TEST", yte, lr.predict(Xte))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

SBERT kNN k=3: val macro F1 0.4276
SBERT kNN k=5: val macro F1 0.4438
SBERT kNN k=10: val macro F1 0.4721
SBERT kNN k=20: val macro F1 0.5042
SBERT kNN k=50: val macro F1 0.5189

=== SBERT kNN (k=50) - TEST ===
Accuracy: 0.6186 | Macro F1: 0.4834
                        precision    recall  f1-score   support

                 False     0.7869    0.7619    0.7742       126
          Mostly False     0.4054    0.3846    0.3947        39
Partly True/Misleading     0.2571    0.3103    0.2812        29

              accuracy                         0.6186       194
             macro avg     0.4831    0.4856    0.4834       194
          weighted avg     0.6310    0.6186    0.6242       194

[[96 16 14]
 [12 15 12]
 [14  6  9]]
SBERT + LR C=0.1: val macro F1 0.4758
SBERT + LR C=1: val macro F1 0.5164
SBERT + LR C=10: val macro F1 0.4852
SBERT + LR C=100: val macro F1 0.4565

=== SBERT + LogisticRegression (C=1) - TEST ===
Accuracy: 0.6392 | Macro F1: 0.5662
                        precisi

In [22]:
# --------------------------------------------------
# 4. Website artifacts
#    Retrieval corpus = ALL rows (including the unlabeled ones: they still have
#    titles, links and evidence worth showing as "similar fact-checks").
# --------------------------------------------------
corpus = df.copy()
corpus["title"] = corpus["title"].fillna("")
corpus["link"] = corpus["link"].fillna("")
corpus["evidence"] = corpus["evidence"].fillna("").astype(str)
corpus["label_name"] = corpus["label_id"].map(lambda x: ID2LABEL[int(x)] if pd.notna(x) else "unlabeled")
corpus["evidence_snippet"] = corpus["evidence"].str.slice(0, 600)
corpus["index_text"] = (corpus["title"] + " | " + corpus["claim"].astype(str)).str.strip(" |")
 
corpus[["claim", "title", "link", "source_dataset", "label_name",
        "evidence_snippet", "index_text"]].to_csv(f"{ART}/corpus.csv", index=False)
np.save(f"{ART}/corpus_emb.npy", enc(corpus["index_text"]))
 
# final LR for the website: train + val
lr_final = LogisticRegression(C=best_C, max_iter=3000, class_weight="balanced")
lr_final.fit(np.vstack([Xtr, Xva]), np.concatenate([ytr, yva]))
joblib.dump(lr_final, f"{ART}/sbert_lr.joblib")
 
# MuRIL model (skip intermediate checkpoints to keep the bundle small)
shutil.copytree(MURIL_DIR, f"{ART}/muril", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("checkpoint-*", "optimizer.pt", "*.bin.index.json"))
 
shutil.make_archive("/kaggle/working/artifacts_bundle", "zip", ART)
print("\nDone. Download /kaggle/working/artifacts_bundle.zip and unzip it next to app.py as ./artifacts")

Batches:   0%|          | 0/94 [00:00<?, ?it/s]


Done. Download /kaggle/working/artifacts_bundle.zip and unzip it next to app.py as ./artifacts


In [23]:
   from sklearn.model_selection import StratifiedKFold, cross_val_predict
   X_all = np.vstack([Xtr, Xva, Xte]); y_all = np.concatenate([ytr, yva, yte])
   skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
   p = cross_val_predict(LogisticRegression(C=1, max_iter=3000, class_weight="balanced"), X_all, y_all, cv=skf)
   print(classification_report(y_all, p, target_names=list(ID2LABEL.values()), digits=4))

                        precision    recall  f1-score   support

                 False     0.8785    0.6561    0.7511      1256
          Mostly False     0.4685    0.7635    0.5806       389
Partly True/Misleading     0.2802    0.3505    0.3115       291

              accuracy                         0.6317      1936
             macro avg     0.5424    0.5900    0.5477      1936
          weighted avg     0.7062    0.6317    0.6508      1936

